# DataMate — clone → download model → GPU inference3000 → evaluate3000

Chọn **Runtime → Change runtime type → GPU**. Notebook dùng Qwen2.5-1.5B Q4_K_M tải từ
revision cố định, load bằng llama-cpp-python CUDA trong Python3.11 `.venv` riêng.
**Không gọi API inference, không cần upload weights hoặc wheel.**

Chuẩn bị `train.zip` từ tập train gốc (data/, questions.jsonl, answers.jsonl, rulebook.md,
calendar.csv), đặt tại **MyDrive/DataMate/train.zip**. Dataset không nằm trong GitHub theo
chính sách repository. Repo private cần Colab Secret `GITHUB_TOKEN` có quyền Contents:read
cho đúng repo; token chỉ dùng clone, không ghi vào notebook, URL Git hay source.

Source S1–S5 phải đã được commit/push lên ref được chọn. S6 là các bước kiểm chứng/evaluation
bên dưới; không phải metric mới và không tự được chứng nhận chỉ vì source có mặt.
Phạm vi hữu hạn xem docs/capability-matrix.md. Colab không thay nghiệm thu Docker/Blackwell/SLA.

Theo yêu cầu người dùng: inference đủ3000 trước, rồi chấm **tất cả3000**. Các partition giữ
membership cũ nhưng1224 câu trước đây giữ lại sẽ được chấm; không gọi kết quả là blind holdout.
Checkpoint theo100 câu giúp chạy tiếp sau ngắt phiên; dùng3000 nếu muốn một process liên tục.


In [ ]:
import getpass
import json
import os
import shutil
import signal
import subprocess
import sys
import tempfile
import time
from pathlib import Path

from google.colab import drive, userdata

REPO_URL = "https://github.com/TruongQuangPhat/Zalo-AI-Challenge.git"
REPO_REF = "main"  # Branch/tag/SHA containing S3–S5 source and scripts/colab_pipeline.py.
PROJECT = Path("/content/Zalo-AI-Challenge")
DRIVE_ROOT = Path("/content/drive/MyDrive/DataMate")
TRAIN_ARCHIVE = DRIVE_ROOT / "train.zip"
RUN_ID = "baseline-full-train-v1"  # Keep for resume; change for independent repetitions.
CHUNK_SIZE = 100  # 3000 = one process; all settings are locked into checkpoint identity.

drive.mount("/content/drive")
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
if not PROJECT.exists():
    try:
        token = userdata.get("GITHUB_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = getpass.getpass("GitHub token (read-only Contents for this private repo): ")
    clone_env = os.environ.copy()
    clone_env.update(DATAMATE_GITHUB_TOKEN=token, GIT_TERMINAL_PROMPT="0")
    try:
        with tempfile.TemporaryDirectory() as temporary:
            askpass = Path(temporary) / "askpass"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os,sys\n"
                'print("x-access-token" if "username" in sys.argv[1].lower() '
                'else os.environ["DATAMATE_GITHUB_TOKEN"])\n'
            )
            askpass.chmod(0o700)
            clone_env["GIT_ASKPASS"] = str(askpass)
            subprocess.run(
                ["git", "-c", "credential.helper=", "clone", REPO_URL, str(PROJECT)],
                env=clone_env,
                check=True,
            )
        subprocess.run(["git", "checkout", "--detach", REPO_REF], cwd=PROJECT, check=True)
    finally:
        clone_env.pop("DATAMATE_GITHUB_TOKEN", None)
        del token, clone_env
else:
    remote = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True
    ).strip()
    assert remote == REPO_URL, "Existing project points to another repo; use a fresh runtime."
    subprocess.run(
        [
            "git",
            "diff",
            "--exit-code",
            "--",
            "src",
            "scripts",
            "pyproject.toml",
            "uv.lock",
            "predict.sh",
        ],
        cwd=PROJECT,
        check=True,
    )

required = [
    "scripts/colab_pipeline.py",
    "src/agent/business_execution.py",
    "src/agent/entities.py",
    "src/data/normalization/entities.py",
    "src/semantics/business_rules.py",
    "docs/model-license.txt",
]
assert all((PROJECT / name).is_file() for name in required), (
    "Remote ref is missing current S3–S5/Colab source. Commit/push it before this clone workflow."
)
print(
    "Source SHA:",
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip(),
)

In [ ]:
# The Colab kernel can use another Python; every project command uses isolated Python3.11.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.12.0"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
subprocess.run(
    [
        "uv",
        "sync",
        "--locked",
        "--python",
        "3.11",
        "--no-default-groups",
        "--group",
        "dev",
        "--group",
        "llm",
        "--no-install-package",
        "llama-cpp-python",
    ],
    cwd=PROJECT,
    check=True,
)
PYTHON = PROJECT / ".venv/bin/python"
ARTIFACTS = PROJECT / ".artifacts"
ARTIFACTS.mkdir(exist_ok=True)
MODEL_DIR = ARTIFACTS / "models/qwen2.5-1.5b-instruct"
MODEL = MODEL_DIR / "model.gguf"

# Stream GGUF + upstream prebuilt CUDAwheel, verify SHA before use, install locked CUDA libs.
subprocess.run(
    [str(PYTHON), "-m", "scripts.colab_pipeline", "setup", "--directory", str(MODEL_DIR)],
    cwd=PROJECT,
    check=True,
)

In [ ]:
# Quality gates use only repository/synthetic tests, not native answer feedback.
quality_log = ARTIFACTS / "colab-quality.txt"
quality_commands = [
    [str(PYTHON), "-m", "ruff", "check", "src", "scripts", "tests"],
    [str(PYTHON), "-m", "ruff", "format", "--check", "src", "scripts", "tests"],
    ["uv", "lock", "--check", "--offline"],
    [str(PYTHON), "-m", "pytest", "-q", "--basetemp", str(ARTIFACTS / "pytest-colab")],
]
with quality_log.open("w") as log:
    for command in quality_commands:
        result = subprocess.run(command, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
        if result.returncode:
            print(quality_log.read_text()[-8000:])
            raise RuntimeError("Quality gate failed; stop before inference.")
print(quality_log.read_text()[-2000:])
shutil.rmtree(ARTIFACTS / "pytest-colab", ignore_errors=True)

In [ ]:
# Copy ZIP once, extract on local SSD. Staging never parses or copies answers into inference.
assert TRAIN_ARCHIVE.is_file(), "Place original train.zip in MyDrive/DataMate first."
LOCAL_ZIP = ARTIFACTS / "train.zip"
if not LOCAL_ZIP.exists():
    shutil.copyfile(TRAIN_ARCHIVE, LOCAL_ZIP)
RAW = ARTIFACTS / "raw-train"
if not RAW.exists():
    subprocess.run(
        [
            str(PYTHON),
            "-m",
            "scripts.colab_pipeline",
            "extract",
            "--archive",
            str(LOCAL_ZIP),
            "--destination",
            str(RAW),
        ],
        cwd=PROJECT,
        check=True,
    )
roots = [p.parent for p in RAW.rglob("questions.jsonl")]
assert len(roots) == 1
TRAIN = roots[0]
DATA = ARTIFACTS / "inference/train"
subprocess.run(
    [
        str(PYTHON),
        "-m",
        "scripts.colab_pipeline",
        "stage",
        "--train-dir",
        str(TRAIN),
        "--data-dir",
        str(DATA),
    ],
    cwd=PROJECT,
    check=True,
)
assert not (DATA / "answers.jsonl").exists()
QUESTIONS = [json.loads(line) for line in (DATA / "questions.jsonl").read_text().splitlines()]
assert len(QUESTIONS) == len({q["id"] for q in QUESTIONS}) == 3000

In [ ]:
def run_command(command):
    # Terminate only this notebook's process group on interruption; helper cleans GPU children.
    process = subprocess.Popen(command, cwd=PROJECT, start_new_session=True)
    try:
        if process.wait() != 0:
            raise RuntimeError("Command failed; keep logs and do not proceed to scoring.")
    finally:
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()


# Smoke proves actual model loading/offload, not correctness. No answers are opened.
SMOKE = ARTIFACTS / ("colab-smoke-" + str(time.time_ns()))
SMOKE_CACHE = ARTIFACTS / "colab-smoke-cache"
smoke_ids = [q["id"] for q in QUESTIONS[:3]]
env_code = (
    "from scripts.colab_pipeline import cuda_environment; "
    "import json; print(json.dumps(cuda_environment()))"
)
cuda_env = json.loads(
    subprocess.check_output([str(PYTHON), "-c", env_code], cwd=PROJECT, text=True)
)
command = [
    "bash",
    "predict.sh",
    "--data-dir",
    str(DATA),
    "--output-dir",
    str(SMOKE),
    "--cache-dir",
    str(SMOKE_CACHE),
    "--model",
    str(MODEL),
    "--gpu-layers",
    "-1",
    "--model-threads",
    "4",
    "--question-timeout",
    "120",
]
for qid in smoke_ids:
    command += ["--question-id", qid]
process = subprocess.Popen(command, cwd=PROJECT, env=cuda_env, start_new_session=True)
try:
    assert process.wait() == 0, "Smoke failed; stop."
finally:
    if process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait(timeout=20)
check = (
    "import json,sys; from pathlib import Path; "
    "from scripts.colab_pipeline import validate_part,remove_prepared_copies; "
    'print(validate_part(Path(sys.argv[1]),json.loads(sys.argv[2]),"smoke")); '
    "remove_prepared_copies(Path(sys.argv[1]),Path(sys.argv[3]))"
)
subprocess.run(
    [str(PYTHON), "-c", check, str(SMOKE), json.dumps(smoke_ids), str(SMOKE_CACHE)],
    cwd=PROJECT,
    env=cuda_env,
    check=True,
)
print("Smoke verified: local GGUF, real29layers, no silent CPU fallback.")

Cell tiếp chạy mọi ID, kể cả unsupported. CHUNK_SIZE=100 lưu checkpoint sau từng phần hoàn tất;
CHUNK_SIZE=3000 chạy một process. Giữ cùng RUN_ID/source/config khi resume; checkpoint khác
checksum/source bị từ chối. Không sửa draft/output. SQLite/cache ở SSD/content, Drive chỉ lưu kết quả.
Smoke dùng cache riêng; thời gian full run ghi cả loading/preparation và các lần reload theo chunk.
Không suy SLA BTC từ Colab. Chưa mở/parse gold ở các cell phía trên.


In [ ]:
RUN = ARTIFACTS / "colab-runs" / RUN_ID
CACHE = ARTIFACTS / "colab-cache"
CHECKPOINTS = DRIVE_ROOT / "results" / RUN_ID
run_command(
    [
        str(PYTHON),
        "-m",
        "scripts.colab_pipeline",
        "run",
        "--data-dir",
        str(DATA),
        "--output-dir",
        str(RUN),
        "--cache-dir",
        str(CACHE),
        "--model",
        str(MODEL),
        "--checkpoint-dir",
        str(CHECKPOINTS),
        "--chunk-size",
        str(CHUNK_SIZE),
    ]
)
print("Inference complete3000. Predictions:", RUN / "submission.jsonl")

In [ ]:
# First label-reading cell: scorer itself enforces completed3000/checksums/source guard.
REPORT = PROJECT / "reports/stage_results.json"
run_command(
    [
        str(PYTHON),
        "-m",
        "scripts.colab_pipeline",
        "evaluate",
        "--answers",
        str(TRAIN / "answers.jsonl"),
        "--run-dir",
        str(RUN),
        "--report",
        str(REPORT),
        "--evaluation-id",
        RUN_ID,
    ]
)
shutil.copyfile(REPORT, CHECKPOINTS / "stage_results.json")
shutil.copyfile(quality_log, CHECKPOINTS / "quality.txt")
shutil.copyfile(RUN / "evaluation-verification.json", CHECKPOINTS / "evaluation-verification.json")
shutil.copyfile(RUN / "selection.json", CHECKPOINTS / "selection.json")
shutil.copyfile(MODEL_DIR / "setup.json", CHECKPOINTS / "setup.json")
print("Results and evidence saved:", CHECKPOINTS)

Kết quả câu hỏi dùng chung `reports/stage_results.json`: S1–S5 buckets được khóa từ lexical
proposal trước gold, S6 rollup đủ3000; không cộng rollup/repetitions vào mẫu số. Lịch sử S1/S2
và fixtures trước đó giữ nguyên. Correct/wrong/abstain theo scorer; expected abstain vẫn0 điểm.
Không coi behavior pass hoặc đủ3000 dòng là gold correct. Chi tiết kỹ thuật ở .artifacts/Drive,
không trộn hashes/logs/drafts vào JSON kết quả. Pipeline không tự commit/push hoặc nộp thi.

Nguồn tải: [Qwen pinned revision](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/tree/91cad51170dc346986eccefdc2dd33a9da36ead9),
[llama-cpp-python CUDA0.3.16](https://github.com/abetlen/llama-cpp-python/releases/tag/v0.3.16-cu124).
[Colab FAQ](https://research.google.com/colaboratory/faq.html): GPU/thời lượng không bảo đảm;
checkpoint giúp phục hồi nhưng không né giới hạn dịch vụ. Notebook chưa được thực thi trên Colab
bởi agent trong phiên phát triển; phải đọc output các gate thực tế trước khi tuyên bố nghiệm thu.
